## Introduction
This notebook collects Pokemon data from PokeAPI. In it, I retrieve two Pokemon from the API, Charizard and Bulbasaur, check whether their data is complete and reasonable, and organize their "sprites" (for Charizard) and "abilities" (for Bulbasaur) fields into pandas DataFrames for analysis.

## API Fixture Importing

In [67]:
from pathlib import Path
import json
import pandas as pd

CHARIZARD_ENDPOINT = "https://pokeapi.co/api/v2/pokemon/charizard" 
BULBASAUR_ENDPOINT = "https://pokeapi.co/api/v2/pokemon/bulbasaur"

DATA_DIR = Path("./")
FIXTURE_DIR = DATA_DIR / "api-fixtures"
assert FIXTURE_DIR.exists(), "Path error."
charizard_path = FIXTURE_DIR / "charizardapi.json"
bulbasaur_path = FIXTURE_DIR / "bulbasaurapi.json"

try:
    with charizard_path.open(encoding="utf-8") as charizardhandle:
        fixturecharizard = json.load(charizardhandle)
    with bulbasaur_path.open(encoding="utf-8") as bulbasaurhandle:
        fixturebulbasaur = json.load(bulbasaurhandle)

except FileNotFoundError as exc: # https://docs.python.org/3/builtins/exceptions.html
    raise FileNotFoundError(f"Could not find the fixture file: {charizard_path} or {bulbasaur_path}") from exc

except UnicodeDecodeError as exc: # https://docs.python.org/3/builtins/exceptions.html
    raise ValueError("The fixture file uses an unsupported encoding.") from exc

except json.JSONDecodeError as exc: # https://docs.python.org/3/library/json.html
    raise ValueError(f"The fixture file is not valid JSON: {charizard_path} or {bulbasaur_path}") from exc
    
print(fixturecharizard)
print(fixturecharizard.keys())
print(fixturebulbasaur)
print(fixturebulbasaur.keys())

{'id': 6, 'name': 'charizard', 'base_experience': 240, 'height': 17, 'is_default': True, 'order': 7, 'weight': 905, 'abilities': [{'is_hidden': False, 'slot': 1, 'ability': {'name': 'blaze', 'url': 'https://pokeapi.co/api/v2/ability/66/'}}, {'is_hidden': True, 'slot': 3, 'ability': {'name': 'solar-power', 'url': 'https://pokeapi.co/api/v2/ability/94/'}}], 'past_abilities': [{'generation': {'name': 'generation-iv', 'url': 'https://pokeapi.co/api/v2/generation/4/'}, 'abilities': [{'is_hidden': True, 'slot': 3, 'ability': None}]}], 'forms': [{'name': 'charizard', 'url': 'https://pokeapi.co/api/v2/pokemon-form/6/'}], 'game_indices': [{'game_index': 180, 'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}}, {'game_index': 180, 'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}}, {'game_index': 180, 'version': {'name': 'yellow', 'url': 'https://pokeapi.co/api/v2/version/3/'}}, {'game_index': 6, 'version': {'name': 'gold', 'url': 'https://pokeapi.

I first load fixture files for both pokemon so the notebook can be reproduced even when live API info is not used, with many error checks to make sure everything works. The keys for these fixtures include things like the id, name, base experience, height, weight, and abilities of the selected pokemon.

## Payload Validation

In [68]:
EXPECTED_FIELDS = {"id", "name", "height", "weight"}

def validate_payload(payload): # checks the structure
    if not isinstance(payload, dict): # seeing if payload is a dict
        raise TypeError("API payload must be a JSON object")
    if "sprites" not in payload or not isinstance(payload["sprites"], dict): # checking if the key i want in the payload (sprites) exists
        raise ValueError("Missing sprite object") 
    missing = EXPECTED_FIELDS - set(payload)
    if missing: # checking if any fields in the keys list (from fixture) arent in the live copy
        raise ValueError(f"Missing Pokemon fields: {sorted(missing)}")
    return payload

def validate_types(payload): # checks the content
    if "abilities" not in payload or not isinstance(payload["abilities"], list): # checking if the key i want in the payload (abilities) exists
        raise ValueError("Missing abilities object")
    if len(payload["abilities"]) != 2: # checks if the amount of abilities is exactly 2 for bulbasaur
        raise ValueError("Bulbasaur should have exactly 2 abilities")
    if payload["height"] <= 0 or payload["weight"] <= 0 or payload["id"] <= 0:
        raise ValueError("Pokemon height, weights, and/or id should be positive")
    return payload

Here I create two validation functions: the first is for the structure of the fixtures and making sure that is in order, while the second is for the content of the fixtures and making sure that is in order. These will be called shortly.

## Live API Payload Importing

In [69]:
def acquire_charizard(use_live=False):
  source = "cached fixture"
  payload = fixturecharizard
  if use_live:
    try:
      import requests
      response = requests.get(CHARIZARD_ENDPOINT, timeout=10) # purpose of timeout: to stop trying to get the data if its already not working
      response.raise_for_status() # checks for all http codes: 2xx codes are okay, while 4xx and 5xx codes return an error
      payload = response.json() # converts the response to json
      validate_payload(payload) # validation code
      source = response.url # url of the source
    except (requests.RequestException, ValueError, TypeError) as exc:
      print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
      payload = fixturecharizard
  return validate_payload(payload), source

def acquire_bulbasaur(use_live=False):
  source = "cached fixture"
  payload = fixturebulbasaur
  if use_live:
    try:
      import requests
      response = requests.get(BULBASAUR_ENDPOINT, timeout=10) # purpose of timeout: to stop trying to get the data if its already not working
      response.raise_for_status() # checks for all http codes: 2xx codes are okay, while 4xx and 5xx codes return an error
      payload = response.json() # converts the response to json
      validate_payload(payload) # validation code
      source = response.url # url of the source
    except (requests.RequestException, ValueError, TypeError) as exc:
      print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
      payload = fixturebulbasaur
  return validate_payload(payload), source
  

charizard_payload, charizard_source = acquire_charizard(use_live=True)
bulbasaur_payload, bulbasaur_source = acquire_bulbasaur(use_live=True)
validate_types(charizard_payload)
validate_types(bulbasaur_payload)

{'id': 1,
 'name': 'bulbasaur',
 'base_experience': 64,
 'height': 7,
 'is_default': True,
 'order': 1,
 'weight': 69,
 'abilities': [{'is_hidden': False,
   'slot': 1,
   'ability': {'name': 'overgrow',
    'url': 'https://pokeapi.co/api/v2/ability/65/'}},
  {'is_hidden': True,
   'slot': 3,
   'ability': {'name': 'chlorophyll',
    'url': 'https://pokeapi.co/api/v2/ability/34/'}}],
 'past_abilities': [{'generation': {'name': 'generation-iv',
    'url': 'https://pokeapi.co/api/v2/generation/4/'},
   'abilities': [{'is_hidden': True, 'slot': 3, 'ability': None}]}],
 'forms': [{'name': 'bulbasaur',
   'url': 'https://pokeapi.co/api/v2/pokemon-form/1/'}],
 'game_indices': [{'game_index': 153,
   'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}},
  {'game_index': 153,
   'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}},
  {'game_index': 153,
   'version': {'name': 'yellow',
    'url': 'https://pokeapi.co/api/v2/version/3/'}},
  {'game_in

Here I make functions for both Pokemon to import the live API, using the fixture data as a fallback if the request fails. After validating the Pokémon responses and using those earlier functions inside of these functions, I organize selected fields into DataFrames, including the "sprites" field for Charizard and "abilities" field for Bulbasaur, which I end up using primarily. The abilities dataframe has one row per ability. I extract the ability name and URL into separate columns and remove the original nested dictionary so the table is easier to analyze.

## DataFrame Creation, Editing, & Final Validation

In [ ]:
sprites = pd.DataFrame(charizard_payload["sprites"])
sprites

abilities = pd.DataFrame(bulbasaur_payload["abilities"])
abilities["ability_name"] = [ability["name"] for ability in abilities["ability"]]
abilities["ability_url"] = [ability["url"] for ability in abilities["ability"]]
abilities = abilities.drop(columns=["ability"])
abilities

print(sprites.shape)
print(sprites.dtypes)
print(sprites.isna().sum())

print(abilities.shape)
print(abilities.dtypes)
print(abilities.isna().sum())
print("Duplicate ability slots:", abilities["slot"].duplicated().sum())

(13, 10)
other                 object
versions              object
back_shiny               str
back_female           object
front_shiny              str
back_default             str
front_female          object
front_default            str
back_shiny_female     object
front_shiny_female    object
dtype: object
other                  9
versions               4
back_shiny             0
back_female           13
front_shiny            0
back_default           0
front_female          13
front_default          0
back_shiny_female     13
front_shiny_female    13
dtype: int64
(2, 4)
is_hidden        bool
slot            int64
ability_name      str
ability_url       str
dtype: object
is_hidden       0
slot            0
ability_name    0
ability_url     0
dtype: int64
Duplicate ability slots: 0


The validation checks show the shape and data types of the "sprites" and "abilities" DataFrames. The missing-value results show whether any fields are incomplete. The duplicate ability-slot check shows whether any ability positions are repeated. The range checks show that the Pokémon IDs, heights, and weights are positive.

The abilities DataFrame has two rows, one for each ability, and the duplicate slot check shows that the ability slots are not repeated. The "sprites" DataFrame contains missing values because not every sprite variant is available, which is documented.